# Configuração do Ambiente

In [0]:
########################################################
#### LIBS
########################################################
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    BooleanType, DoubleType, TimestampType, LongType
)
from pyspark.sql.window import Window
import pyspark
#from pyspark.sql.functions import col
import pandas as pd
import numpy as np
import requests
import json
import os
import hashlib
import json as _json
from datetime import datetime, timezone

# Leitura dos dados

In [0]:
### função de leitura dos dados
def leitura_prata(dados):
     return spark.read.format("parquet").load(f"/Volumes/workspace/default/inep_avaliacao_alfabetizacao/silver/{dados}")

In [0]:
### leitura dos dados
alunos = leitura_prata("alunos")
municipio = leitura_prata("municipio")
uf = leitura_prata("uf")
meta_municipio = leitura_prata("meta_alfabetizacao_municipio")
meta_uf = leitura_prata("meta_alfabetizacao_uf")
meta_brasil = leitura_prata("meta_alfabetizacao_brasil")

ibge = leitura_prata("ibge_ideb")


# Construção da camada Ouro

## Município

Note que, os municípios se repetem. Dessa forma, a análise será feita **apenas** com as redes municipais de ensino.

In [0]:
###
dados_final = (
    municipio
    .filter(F.col("ano") == "2023")
    .groupBy("id_municipio")
    .count()
    .withColumnRenamed("count", "n")
    .orderBy(F.col("n").desc())
    .limit(5)
)

dados_final.show()

In [0]:
municipio_alvo = (
    municipio
    .filter((F.col("ano") == 2024) & (F.col("rede") == "3"))
    .select("id_municipio", F.col("media_portugues").alias("Y"))
    .withColumn("ALVO", F.when(F.col("Y") >= 743, 1).otherwise(0))
)

municipio_alvo.show(5)
municipio_alvo.count()

In [0]:
municipio_preditores = (
    municipio
    .filter((F.col("ano") == 2023) & (F.col("rede") == "3"))
    .select("id_municipio", "taxa_alfabetizacao", "media_portugues")
    .withColumn(
        "nivel",
        F.when(F.col("media_portugues") < 650, 0)
         .when((F.col("media_portugues") >= 650) & (F.col("media_portugues") < 675), 1)
         .when((F.col("media_portugues") >= 675) & (F.col("media_portugues") < 700), 2)
         .when((F.col("media_portugues") >= 700) & (F.col("media_portugues") < 725), 3)
         .when((F.col("media_portugues") >= 725) & (F.col("media_portugues") < 750), 4)
         .when((F.col("media_portugues") >= 750) & (F.col("media_portugues") < 775), 5)
         .when((F.col("media_portugues") >= 775) & (F.col("media_portugues") < 800), 6)
         .when((F.col("media_portugues") >= 800) & (F.col("media_portugues") < 825), 7)
         .when(F.col("media_portugues") > 825, 8)
         .otherwise(None)  
    )
)

municipio_preditores.show(5)
municipio_preditores.count()

In [0]:
municipio_preditores = municipio_preditores.select(
    "id_municipio",
    *[F.col(c).alias(f"{c}_2023") for c in municipio_preditores.columns if c != "id_municipio"]
)

municipio_final = municipio_alvo.join(municipio_preditores, on = "id_municipio", how = "left")
municipio_final.show()
municipio_final.count()

In [0]:
municipio_final.filter(F.col("id_municipio").isNull()).show()

## Meta municipal

In [0]:
###
meta_municipio2 = (meta_municipio
    .filter(F.col("ano") == 2023)
    .select("id_municipio", "meta_alfabetizacao_2024", "meta_alfabetizacao_2030"))

meta_municipio2.show(5)
meta_municipio2.count()

In [0]:
meta_municipio2.filter(F.col("id_municipio").isNull()).show()

In [0]:
### rede municipal na tabela de metas (aceita o código "3" ou o nome "Municipal")
filtro_rede_municipal = F.lower(F.trim(F.col("rede").cast("string"))).isin("3", "municipal")

if meta_municipio2.filter(filtro_rede_municipal).limit(1).count() > 0:
    meta_municipal = meta_municipio2.filter(filtro_rede_municipal)
else:
    redes_meta = [r["rede"] for r in meta_municipio2.select("rede").distinct().collect()]
    print(f"Atenção: nenhuma linha de rede municipal encontrada na meta. Redes disponíveis: {redes_meta}")
    print("A tabela de metas será usada sem filtro de rede.")
    meta_municipal = meta_municipio2

meta_municipal = meta_municipal.select("id_municipio", "meta_alfabetizacao_2024", "meta_alfabetizacao_2030")

### checagem: uma linha por município
n_linhas_meta = meta_municipal.count()
n_mun_meta = meta_municipal.select("id_municipio").distinct().count()
print(f"Meta municipal: {n_linhas_meta} linhas | {n_mun_meta} municípios distintos")

if n_linhas_meta != n_mun_meta:
    raise ValueError(
        "A tabela de metas tem mais de uma linha por município. "
        "Confira os valores de 'rede' na célula anterior antes de seguir para o join."
    )

## Dados de fontes externas

A tabela `ibge` (camada prata) foi construída por um *left join* do IBGE com o IDEB. Por isso ela contém **uma linha por município e por rede do IDEB** (Estadual, Municipal, Pública...) e, para os municípios **sem resultado no IDEB**, uma única linha com `rede` nula.

Filtrar `rede == "Municipal"` diretamente descarta esses municípios sem IDEB e, com eles, a área, a população e o PIB, que não dependem da rede. Esse era o motivo das muitas colunas nulas após o *join*.

Assim, a tabela é separada em duas partes:
- **dados do município** (área, população, PIB, região): uma linha por município, independentemente da rede;
- **dados do IDEB da rede municipal** (`matematica_saeb`, `portugues_saeb`, `ideb_2023`).

As duas partes são unidas por um *left join*, mantendo todos os municípios do IBGE. A coluna `sigla_uf` do IDEB não é usada, pois fica nula para os municípios sem IDEB; ela é recriada a partir do código do município na etapa de imputação.

In [0]:
### dados do município: não dependem da rede (uma linha por município)
ibge_municipio = (ibge
    .select("CD_MUN", "NM_MUN", "NM_REGIAO", "AREA_KM2", "POPULACAO", "pib")
    .dropDuplicates(["CD_MUN"])
)

### dados do IDEB: somente a rede municipal
ideb_municipal = (ibge
    .filter(F.lower(F.trim(F.col("rede"))) == "municipal")
    .select("CD_MUN", "matematica_saeb", "portugues_saeb", "ideb_2023")
    .dropDuplicates(["CD_MUN"])
)

### junção das duas partes
ibge2 = (ibge_municipio
    .join(ideb_municipal, on="CD_MUN", how="left")
    .withColumnsRenamed({
        "CD_MUN": "id_municipio",
        "AREA_KM2": "area_km2",
        "POPULACAO": "pop",
        "NM_MUN":"nome_municipio",
        "NM_REGIAO":"nome_regiao",
        "portugues_saeb":"portugues_saeb_2023",
        "matematica_saeb":"matematica_saeb_2023",
        })
)

ibge2.show(5)

In [0]:
### checagens
n_mun_ibge = ibge2.count()
n_mun_ibge_distintos = ibge2.select("id_municipio").distinct().count()
n_mun_com_ideb = ibge2.filter(F.col("ideb_2023").isNotNull()).count()

print(f"Municípios no IBGE: {n_mun_ibge} (distintos: {n_mun_ibge_distintos})")
print(f"Municípios com IDEB 2023 da rede municipal: {n_mun_com_ideb}")
print(f"Municípios sem IDEB 2023 da rede municipal: {n_mun_ibge - n_mun_com_ideb}")
print("O Distrito Federal não tem rede municipal de ensino; ele sai da base no join, pois municipio2 contém apenas a rede municipal.")

## Join das tabelas

A tabela `municipio2` é a base (contém a variável `taxa_alfabetizacao`), por isso é usada em um *left join* com as demais. Todas as tabelas já estão restritas à **rede municipal** e ao ano de **2023**, e têm **uma linha por município**, de modo que a junção é feita apenas pela chave `id_municipio`:
- `meta_municipal`: metas de alfabetização de 2024 e 2030;
- `ibge2`: dados do IBGE, PIB e IDEB da rede municipal.

A chave `id_municipio` é padronizada nas três tabelas (texto, sem espaços e sem o sufixo `.0` que pode surgir quando o código passa por um tipo numérico). As colunas `ano` e `rede` de `municipio2` são removidas, pois têm valor único após os filtros.

In [0]:
### padronização da chave
def padronizar_id_municipio(df):
    return df.withColumn(
        "id_municipio",
        F.regexp_replace(F.trim(F.col("id_municipio").cast("string")), r"\.0$", "")
    )

### join das três tabelas
base = padronizar_id_municipio(municipio_final).drop("ano", "rede")

dados_ouro = (base
    .join(padronizar_id_municipio(meta_municipal), on="id_municipio", how="left")
    .join(padronizar_id_municipio(ibge2), on="id_municipio", how="left")
)

dados_ouro.show(5)

In [0]:
### verificação do join
n_base = base.count()
n_ouro = dados_ouro.count()
n_mun_ouro = dados_ouro.select("id_municipio").distinct().count()

print(f"Linhas em municipio2: {n_base}")
print(f"Linhas em dados_ouro: {n_ouro} | municípios distintos: {n_mun_ouro}")

if n_ouro != n_base:
    raise ValueError("O join alterou o número de linhas: alguma tabela tem mais de uma linha por município.")

### municípios da base sem correspondência em cada tabela
sem_meta = base.join(padronizar_id_municipio(meta_municipal), on="id_municipio", how="left_anti")
sem_ibge = base.join(padronizar_id_municipio(ibge2), on="id_municipio", how="left_anti")

print(f"Municípios sem meta: {sem_meta.count()}")
print(f"Municípios sem dados do IBGE: {sem_ibge.count()}")

In [0]:
### municípios sem correspondência no IBGE (o esperado é nenhum)
sem_ibge.show()

In [0]:
### quantidade de valores ausentes por coluna após o join
dados_ouro.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in dados_ouro.columns]).show()

## Imputação dos dados

Os valores ausentes das colunas `meta_alfabetizacao_2024` (meta municipal) e `matematica_saeb`, `portugues_saeb` e `ideb_2023` (IDEB) são imputados pela **mediana da UF**.

A UF é obtida pelos dois primeiros dígitos do código IBGE do município (`cd_uf`), e a `sigla_uf` é recriada a partir dele, pois a coluna vinda do IDEB fica nula para os municípios sem correspondência.

Para cada coluna imputada é criada uma coluna indicadora `<coluna>_imputado` (1 = valor imputado, 0 = valor original). Assim, a imputação fica rastreável e o efeito dela pode ser avaliado na modelagem.

Se uma UF não tiver nenhum valor observado em uma coluna, a mediana da UF também é nula. Nesses casos, usa-se a **mediana nacional** como alternativa.

In [0]:
### colunas a serem imputadas
colunas_imputar = ["taxa_alfabetizacao_2023","media_portugues_2023", "meta_alfabetizacao_2024", "meta_alfabetizacao_2030", "matematica_saeb_2023", "portugues_saeb_2023", "ideb_2023"]

### código da UF (2 primeiros dígitos do código IBGE do município) e sigla correspondente
UF_SIGLA = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA", "16": "AP", "17": "TO",
    "21": "MA", "22": "PI", "23": "CE", "24": "RN", "25": "PB", "26": "PE", "27": "AL",
    "28": "SE", "29": "BA", "31": "MG", "32": "ES", "33": "RJ", "35": "SP", "41": "PR",
    "42": "SC", "43": "RS", "50": "MS", "51": "MT", "52": "GO", "53": "DF",
}
ufs = spark.createDataFrame(list(UF_SIGLA.items()), ["cd_uf", "sigla_uf"])

dados_ouro = (dados_ouro
    .drop("cd_uf", "sigla_uf")
    .withColumn("cd_uf", F.substring(F.col("id_municipio"), 1, 2))
    .join(F.broadcast(ufs), on="cd_uf", how="left")
)

print(f"Municípios sem UF identificada: {dados_ouro.filter(F.col('sigla_uf').isNull()).count()}")

### quantidade de valores ausentes antes da imputação
dados_ouro.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in colunas_imputar]).show()

In [0]:
### mediana de cada coluna por UF
medianas_uf = (dados_ouro
    .groupBy("cd_uf", "sigla_uf")
    .agg(*[F.median(c).alias(f"mediana_{c}") for c in colunas_imputar])
)

medianas_uf.orderBy("cd_uf").show(27)

### mediana nacional (alternativa para UFs sem nenhum valor observado)
medianas_brasil = dados_ouro.agg(*[F.median(c).alias(c) for c in colunas_imputar]).first().asDict()
print("Medianas nacionais:", medianas_brasil)

In [0]:
### imputação: valores nulos recebem a mediana da UF (ou a nacional, se a da UF for nula)
dados_ouro = (dados_ouro
    .join(medianas_uf.drop("sigla_uf"), on="cd_uf", how="left")
    .withColumns({f"{c}_imputado": F.col(c).isNull().cast("int") for c in colunas_imputar})
    .withColumns({
        c: F.coalesce(F.col(c), F.col(f"mediana_{c}"), F.lit(medianas_brasil[c]))
        for c in colunas_imputar
    })
    .drop(*[f"mediana_{c}" for c in colunas_imputar])
)

### quantidade de valores ausentes após a imputação (o esperado é zero)
dados_ouro.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in colunas_imputar]).show()

### quantidade de valores imputados por coluna
dados_ouro.select([F.sum(F.col(f"{c}_imputado")).alias(c) for c in colunas_imputar]).show()

dados_ouro.show(5)
print(f"Linhas: {dados_ouro.count()} | Colunas: {len(dados_ouro.columns)}")

In [0]:
dados_ouro.filter(F.col("nivel_2023").isNull()).show()

In [0]:
## adicionar o nível novamente, agora após a imputação dos dados faltantes.
dados_ouro = (
     dados_ouro
     .withColumn(
        "nivel_2023",
        F.when(F.col("media_portugues_2023") < 650, 0)
         .when((F.col("media_portugues_2023") >= 650) & (F.col("media_portugues_2023") < 675), 1)
         .when((F.col("media_portugues_2023") >= 675) & (F.col("media_portugues_2023") < 700), 2)
         .when((F.col("media_portugues_2023") >= 700) & (F.col("media_portugues_2023") < 725), 3)
         .when((F.col("media_portugues_2023") >= 725) & (F.col("media_portugues_2023") < 750), 4)
         .when((F.col("media_portugues_2023") >= 750) & (F.col("media_portugues_2023") < 775), 5)
         .when((F.col("media_portugues_2023") >= 775) & (F.col("media_portugues_2023") < 800), 6)
         .when((F.col("media_portugues_2023") >= 800) & (F.col("media_portugues_2023") < 825), 7)
         .when(F.col("media_portugues_2023") >= 825, 8)
         .otherwise(None)  
    )
)

dados_ouro.show()

In [0]:
###
dados_ouro_imputacao = (dados_ouro
    .select("id_municipio", "sigla_uf","taxa_alfabetizacao_2023_imputado","media_portugues_2023_imputado", 
            "meta_alfabetizacao_2024_imputado", "meta_alfabetizacao_2030_imputado", "matematica_saeb_2023_imputado",
            "portugues_saeb_2023_imputado", "ideb_2023_imputado"))

###
dados_ouro_final = (dados_ouro
    .drop("cd_uf",  "taxa_alfabetizacao_2023_imputado","media_portugues_2023_imputado", 
            "meta_alfabetizacao_2024_imputado", "meta_alfabetizacao_2030_imputado", "matematica_saeb_2023_imputado",
            "portugues_saeb_2023_imputado", "ideb_2023_imputado")
    .withColumnsRenamed({
        "NM_MUN":"nome_mun",
        "NM_REGIAO":"nome_regiao"}))


dados_ouro_final.show(5)
dados_ouro_final.count()

# Enriquecimento da base de dados - criação de novas features/colunas

## Transformação logarítmica

In [0]:
# variáveis que vão receber a transformação logarítmica
colunas_log = ["pop", "pib", "area_km2"]

for coluna in colunas_log:
    dados_ouro_final = dados_ouro_final.withColumn(f"log_{coluna}", F.log1p(coluna))

dados_ouro_final.select(*[f"log_{c}" for c in colunas_log]).show(5)

## Diferença entre as metas

In [0]:
#
dados_ouro_final = (
    dados_ouro_final
    .withColumn("diff_meta_2024", F.round(F.col("taxa_alfabetizacao_2023") - F.col("meta_alfabetizacao_2024"), 2))
    .withColumn("atingiu_meta_2024", (F.col("diff_meta_2024") >= 0).cast("int"))
)

In [0]:
dados_ouro_final.show(5)

# Exportação dos dados

In [0]:
#
file_path =  "/Volumes/workspace/default/inep_avaliacao_alfabetizacao/gold/ouro_modelagem"
#
dados_ouro_final.write.mode("overwrite").parquet(file_path)